In [ ]:
# ============================================================
# Elliptic preprocessing: map txId -> node index, update edgelist,
# handle NaN, standardize features, save clean .pt + .csv files.
# Run on Google Colab. Each "# %%" marks a separate cell.
# ============================================================

# %% Cell 1: Download the 3 CSV files from the shared Drive folder + configuration
from google.colab import auth, drive
auth.authenticate_user()        # pick the Google account that has access to the shared folder
drive.mount('/content/drive')   # used to save the processed outputs to YOUR Drive

import os
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

FOLDER_ID = '18yrerb4P3CO1x4ma3TikcW5l-sxMMbRW'     # shared folder: Explainable-AML-GAT-ResNet/data
DATA_DIR  = '/content/elliptic_data'                 # local copy of the raw files
OUT_DIR   = '/content/drive/MyDrive/elliptic_processed'   # outputs saved to your Drive (persistent)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

REQUIRED = ['elliptic_txs_features.csv', 'elliptic_txs_edgelist.csv', 'elliptic_txs_classes.csv']

if not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in REQUIRED):
    service = build('drive', 'v3')
    resp = service.files().list(
        q=f"'{FOLDER_ID}' in parents and trashed=false",
        fields='files(id, name, size)',
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()
    files = resp.get('files', [])
    if not files:
        raise FileNotFoundError(
            'No files found in the shared folder. Make sure you picked the Google account '
            'that has access to it, then re-run this cell.')
    for f in files:
        if f['name'] not in REQUIRED:
            continue
        path = os.path.join(DATA_DIR, f['name'])
        print('Downloading', f['name'], f.get('size'), 'bytes')
        request = service.files().get_media(fileId=f['id'], supportsAllDrives=True)
        with open(path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request, chunksize=50 * 1024 * 1024)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                print(f'  {int(status.progress() * 100)}%')

missing_files = [f for f in REQUIRED if not os.path.exists(os.path.join(DATA_DIR, f))]
assert not missing_files, f'Missing files: {missing_files}'
print('Raw files ready:', os.listdir(DATA_DIR))

FEATURES_CSV = os.path.join(DATA_DIR, 'elliptic_txs_features.csv')
EDGES_CSV    = os.path.join(DATA_DIR, 'elliptic_txs_edgelist.csv')
CLASSES_CSV  = os.path.join(DATA_DIR, 'elliptic_txs_classes.csv')

# Fit the scaler on training time steps only (1-34) to avoid data leakage.
# Set to False to standardize using the whole dataset.
FIT_ON_TRAIN_ONLY = True
TRAIN_MAX_TIMESTEP = 34

# %% Cell 2: Load data
import numpy as np
import pandas as pd

# features.csv has NO header: col 0 = txId, col 1 = time step, cols 2..166 = 165 features
feat_df = pd.read_csv(FEATURES_CSV, header=None)
edge_df = pd.read_csv(EDGES_CSV)       # has header: txId1, txId2
cls_df  = pd.read_csv(CLASSES_CSV)     # has header: txId, class

feat_df.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(2, feat_df.shape[1])]
print('features:', feat_df.shape, '| edges:', edge_df.shape, '| classes:', cls_df.shape)
assert feat_df.shape == (203769, 167), 'Unexpected number of nodes/features'

# %% Cell 3: Build dictionary mapping txId -> node index (0..N-1)
# Order follows the row order in the features file, so x[i] corresponds to node i.
feat_df['txId']  = feat_df['txId'].astype('int64')
edge_df['txId1'] = edge_df['txId1'].astype('int64')
edge_df['txId2'] = edge_df['txId2'].astype('int64')
cls_df['txId']   = cls_df['txId'].astype('int64')

assert feat_df['txId'].is_unique, 'Duplicate txIds found!'
tx_ids = feat_df['txId'].values
id2idx = {int(t): i for i, t in enumerate(tx_ids)}
N = len(id2idx)
print('Number of nodes N =', N, '| index range: 0 ..', N - 1)

# %% Cell 4: Update edgelist to use node indices
missing = ~(edge_df['txId1'].isin(id2idx) & edge_df['txId2'].isin(id2idx))
print('Edges with txIds not found in features:', int(missing.sum()))
edge_df = edge_df[~missing].reset_index(drop=True)

edge_idx_df = pd.DataFrame({
    'txId1': edge_df['txId1'].map(id2idx).astype('int64'),
    'txId2': edge_df['txId2'].map(id2idx).astype('int64'),
})
assert edge_idx_df.max().max() < N and edge_idx_df.min().min() >= 0
print('New edgelist:', edge_idx_df.shape)
print(edge_idx_df.head())

# %% Cell 5: Labels aligned with node index order
# Original classes: '1' = illicit, '2' = licit, 'unknown'
# Encoding: illicit = 1, licit = 0, unknown = -1
cls_df['class'] = cls_df['class'].astype(str).str.strip()
label_map = {'1': 1, '2': 0, 'unknown': -1}
cls_df['y'] = cls_df['class'].map(label_map)
assert cls_df['y'].notna().all(), 'Unexpected class value found'

y_series = cls_df.set_index('txId')['y'].reindex(tx_ids)
assert y_series.notna().all(), 'Some txIds in features have no label'
y = y_series.values.astype('int64')
print('Label distribution:', dict(zip(*np.unique(y, return_counts=True))), '(-1 unknown, 0 licit, 1 illicit)')

# %% Cell 6: Check and handle NaN / inf values
feat_cols = [c for c in feat_df.columns if c != 'txId']      # time_step + 165 features = 166 columns
X_raw = feat_df[feat_cols].astype('float64')
assert len(feat_cols) == 166

X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
nan_total = int(X_raw.isna().sum().sum())
print('Total NaN/inf values in the 166 columns:', nan_total)

if nan_total > 0:
    print('Columns with NaN:\n', X_raw.isna().sum()[lambda s: s > 0])
    # Fill with column median (robust to outliers)
    X_raw = X_raw.fillna(X_raw.median())
    assert X_raw.isna().sum().sum() == 0

# %% Cell 7: Standardize the 166 features
from sklearn.preprocessing import StandardScaler

time_step = feat_df['time_step'].values.astype('int64')
scaler = StandardScaler()
if FIT_ON_TRAIN_ONLY:
    train_mask_fit = time_step <= TRAIN_MAX_TIMESTEP
    scaler.fit(X_raw.values[train_mask_fit])
    print(f'Scaler fitted on {int(train_mask_fit.sum())} nodes (time step <= {TRAIN_MAX_TIMESTEP})')
else:
    scaler.fit(X_raw.values)
    print('Scaler fitted on the whole dataset')

X = scaler.transform(X_raw.values).astype('float32')
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)   # guards against zero-variance columns
print('X shape:', X.shape, '| mean ~', float(X.mean()), '| std ~', float(X.std()))

# %% Cell 8: Save outputs (.pt + clean .csv)
import json, torch

edge_index = torch.tensor(edge_idx_df.values.T, dtype=torch.long)   # [2, E]
data_dict = {
    'x': torch.tensor(X, dtype=torch.float),             # [N, 166], standardized
    'edge_index': edge_index,                             # [2, E], node indices
    'y': torch.tensor(y, dtype=torch.long),               # -1 unknown, 0 licit, 1 illicit
    'time_step': torch.tensor(time_step, dtype=torch.long),
    'tx_ids': torch.tensor(tx_ids, dtype=torch.long),     # node index i -> original txId
}
torch.save(data_dict, os.path.join(OUT_DIR, 'elliptic_processed.pt'))

# Mapping dictionary (to trace back to original txIds, e.g. for explainability)
with open(os.path.join(OUT_DIR, 'txid_to_idx.json'), 'w') as f:
    json.dump({str(k): v for k, v in id2idx.items()}, f)

# Clean CSV files
edge_idx_df.to_csv(os.path.join(OUT_DIR, 'elliptic_edgelist_indexed.csv'), index=False)

feat_out = pd.DataFrame(X, columns=feat_cols)
feat_out.insert(0, 'node_idx', np.arange(N))
feat_out.insert(1, 'txId', tx_ids)
feat_out['time_step_raw'] = time_step
feat_out['y'] = y
feat_out.to_csv(os.path.join(OUT_DIR, 'elliptic_features_scaled.csv'), index=False)

print('Saved to:', OUT_DIR)
print(os.listdir(OUT_DIR))

# %% Cell 9: Sanity-check the saved .pt file
d = torch.load(os.path.join(OUT_DIR, 'elliptic_processed.pt'))
print({k: tuple(v.shape) for k, v in d.items()})
assert d['x'].shape == (203769, 166)
assert int(d['edge_index'].max()) == 203768 and int(d['edge_index'].min()) == 0
print('OK: data is ready for PyTorch Geometric (Data(x=..., edge_index=..., y=...))')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  100%
  100%
  7%
  15%
  22%
  30%
  38%
  45%
  53%
  60%
  68%
  76%
  83%
  91%
  98%
  100%
Raw files ready: ['elliptic_txs_features.csv', 'elliptic_txs_classes.csv', 'elliptic_txs_edgelist.csv']
features: (203769, 167) | edges: (234355, 2) | classes: (203769, 2)
Number of nodes N = 203769 | index range: 0 .. 203768
Edges with txIds not found in features: 0
New edgelist: (234355, 2)
   txId1  txId2
0      0      1
1      2      3
2      4      5
3      6      7
4      8      9
Label distribution: {np.int64(-1): np.int64(157205), np.int64(0): np.int64(42019), np.int64(1): np.int64(4545)} (-1 unknown, 0 licit, 1 illicit)
Total NaN/inf values in the 166 columns: 0
Scaler fitted on 136265 nodes (time step <= 34)
X shape: (203769, 166) | mean ~ 0.06790158152580261 | std ~ 1.2903838157653809
Saved to: /content/drive/MyDrive/elliptic_processed
['elliptic_proce